In [1684]:
import pandas as pd
import re

# 합계출산율(모의 연령별출산율) 불러오기 -> 확인하기
df = pd.read_csv("시도_합계출산율__모의_연령별_출산율_20260520112924.csv", header=[0, 1])
df.head()

시도별   1993                                                             \
     시도별  합계출산율 모의 연령별출산율:15-19세 20-24세 25-29세 30-34세 35-39세 40-44세 45-49세   
0     전국  1.654              4.4   71.9  176.5   63.2   13.5    2.0    0.2   
1  서울특별시  1.558              2.7   54.2  168.4   70.0   14.0    1.8    0.2   
2  부산광역시  1.505              2.9   58.6  167.3   58.2   11.9    1.7    0.1   
3  대구광역시  1.556              2.8   62.1  174.7   58.6   10.9    1.7    0.2   
4  인천광역시  1.815              6.1   96.4  182.9   62.1   12.8    2.1    0.1   

    1994  ...   2023          2024                                        \
   합계출산율  ... 40-44세 45-49세  합계출산율 모의 연령별출산율:15-19세 20-24세 25-29세 30-34세   
0  1.656  ...    7.9    0.2  0.748              0.4    3.8   20.7   70.4   
1  1.565  ...    8.7    0.2  0.581              0.2    1.3    8.3   49.4   
2  1.468  ...    7.5    0.2  0.683              0.2    2.7   16.5   65.5   
3  1.557  ...    6.9    0.2  0.754              0.3    3.2   19.7   74.5   
4  1.776  ...    7.8    0.1  0.762              0.4    4.6   23.2   70.4   

                        
  35-39세 40-44세 45-49세  
0   46.0    7.7    0.2  
1   46.9    8.9    0.2  
2   44.0    6.8    0.3  
3   45.0    7.0    0.2  
4   45.9    7.8    0.2  

[5 rows x 257 columns]

In [1685]:
# 합계출산율 데이터 전처리 

# 합계출산율 열만 선택
합계출산율 = df.loc[:, df.columns.get_level_values(1) == "합계출산율"]

# 컬럼명을 연도만 남기기(합계출산율 컬럼명 제거)
합계출산율.columns = 합계출산율.columns.get_level_values(0)

# 시도별을 인덱스로 지정
합계출산율.index = df[("시도별", "시도별")]

# 인덱스 이름 "(시도별, 시도별)" -> "시도별"
합계출산율.index.name = "시도별"

# wide 형태 -> long 형태
합계출산율 = 합계출산율.reset_index().melt(
    id_vars="시도별",
    var_name="연도",
    value_name="합계출산율"
)

# 숫자형으로 변환
합계출산율["연도"] = pd.to_numeric(합계출산율["연도"])
합계출산율["합계출산율"] = pd.to_numeric(합계출산율["합계출산율"], errors="coerce")

합계출산율


,시도별,연도,합계출산율
0,전국,1993,1.654
1,서울특별시,1993,1.558
2,부산광역시,1993,1.505
3,대구광역시,1993,1.556
4,인천광역시,1993,1.815
...,...,...,...
571,전북특별자치도,2024,0.808
572,전라남도,2024,1.028
573,경상북도,2024,0.897
574,경상남도,2024,0.820


In [1686]:
# 1인당 지역내총생산, 1인당 민간소비, 1인당 지역총소득, 1인당 개인소득 데이터 불러오기 -> 확인하기
df = pd.read_csv("시도별_1인당_지역내총생산__지역총소득__개인소득_20260520112555.csv", header=[0, 1])
df.head()

시도별       1985       1986       1987       1988       1989       1990  \
     시도별 1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산   
0     전국       2245       2597       2994       3548       3971       4794   
1  서울특별시       2377       2779       3230       3701       4185       5086   
2  부산광역시       1932       2174       2495       2829       3036       3787   
3  대구광역시       1850       2166       2546       2845       3121       3819   
4  인천광역시       2743       3090       3484       4131       4559       5422   

        1991       1992       1993  ...     2020                2021  \
  1인당 지역내총생산 1인당 지역내총생산 1인당 지역내총생산  ... 1인당 개인소득 1인당 민간소비 1인당 지역내총생산   
0       5757       6519       7299  ...    21342    17320      40271   
1       6185       7049       8104  ...    24226    21185      49680   
2       4470       4795       5294  ...    20460    17365      29395   
3       4332       4725       5081  ...    20229    17276      25543   
4       6645       7066       7609  ...    20310    16059      33551   

                                 2022 p)                              
  1인당 지역총소득 1인당 개인소득 1인당 민간소비 1인당 지역내총생산 1인당 지역총소득 1인당 개인소득 1인당 민간소비  
0     40723    22059    18431      41948     42563    23388    20078  
1     54571    24950    22603      51612     57236    26112    24455  
2     31414    21035    18712      31611     32293    22577    20637  
3     29552    20886    18362      26736     31056    22368    19903  
4     35499    21170    17176      35295     37442    22406    18706  

[5 rows x 113 columns]

In [1687]:
# 데이터 전처리 -> 1인당 지역총소득, 1인당 개인소득 분리

# p(provisional). 잠정치 제거
일인당지역총소득및개인소득 = df.drop(columns="2022 p)", level=0)

# 1인당 지역내총생산, 1인당 민간소비 제거 -> 1인당 지역총소득, 1인당 개인소득만 남김
일인당지역총소득및개인소득.drop(columns=["1인당 지역내총생산", "1인당 민간소비"], level=1, inplace=True)

일인당지역총소득및개인소득 = (
    일인당지역총소득및개인소득.set_index(('시도별', '시도별'))   # 지역명을 index로
      .stack(level=0)                  # 연도 level을 행으로 내림
      .reset_index()
      .rename(columns={
          ('시도별', '시도별'): '시도별',
          'level_1': '연도'
      })
)

일인당지역총소득및개인소득.columns.name = None

# 변환 과정에서 stack()이나 reset_index()를 하면 컬럼 순서가 예상과 다르거나, 불필요한 컬럼이 남아있을 수 있음 -> 정리
일인당지역총소득및개인소득 = 일인당지역총소득및개인소득[
    ['시도별', '연도', '1인당 지역총소득', '1인당 개인소득']
]

# 숫자형으로 변환
cols = ["연도", "1인당 지역총소득", "1인당 개인소득"]
일인당지역총소득및개인소득[cols] = 일인당지역총소득및개인소득[cols].apply(
    pd.to_numeric,
    errors="coerce"
)

일인당지역총소득및개인소득


,시도별,연도,1인당 지역총소득,1인당 개인소득
0,전국,2000,13860.0,8694.0
1,전국,2001,14908.0,9120.0
2,전국,2002,16444.0,9820.0
3,전국,2003,17548.0,10389.0
4,전국,2004,18995.0,11063.0
...,...,...,...,...
391,제주특별자치도,2017,30863.0,17816.0
392,제주특별자치도,2018,31193.0,18456.0
393,제주특별자치도,2019,30710.0,18815.0
394,제주특별자치도,2020,30030.0,19954.0


In [1688]:
# 고용률 데이터 불러오기 -> 확인
df = pd.read_csv("고용률_시도__20260520115827.csv")
df.head()

,시도별,성별,1999.06,1999.07,1999.08,1999.09,1999.10,1999.11,1999.12,2000.01,...,2025.07,2025.08,2025.09,2025.10,2025.11,2025.12,2026.01,2026.02,2026.03,2026.04
0,계,계,57.6,57.5,57.3,58.6,59.0,58.8,57.5,56.2,...,63.4,63.3,63.7,63.4,63.4,61.5,61.0,61.8,62.7,63.0
1,계,남자,69.8,69.9,69.8,70.8,71.2,71.2,70.2,68.9,...,71.0,71.0,71.1,70.8,70.9,69.8,68.9,69.3,70.2,70.5
2,계,여자,46.2,46.0,45.6,47.1,47.6,47.2,45.6,44.3,...,56.0,55.8,56.4,56.2,56.0,53.4,53.2,54.6,55.3,55.7
3,서울특별시,계,56.8,56.7,56.5,57.5,57.8,58.4,58.1,57.5,...,62.4,62.3,62.2,61.9,61.8,60.9,60.1,60.8,61.3,61.4
4,서울특별시,남자,68.3,68.1,68.5,69.2,69.7,70.6,70.0,69.6,...,68.9,69.2,68.3,67.9,68.6,67.5,66.2,66.6,67.3,67.2


In [1689]:
# 고용률 데이터 전처리

# 월별 컬럼 찾기 -> 1999.01
month_cols = [
    col for col in df.columns
    if re.match(r"^\d{4}\.\d{2}$", str(col))
]

# 연도 목록 만들기 -> 1999, 2000
years = sorted(set(col[:4] for col in month_cols))

# 시도별, 성별 컬럼은 그대로 두기
고용률 = df[["시도별", "성별"]].copy()

# 같은 연도에 해당하는 12개월 컬럼들의 평균 계산
for year in years:
    # 특정 연도에 해당하는 월별 컬럼들만 골라내기
    year_cols = [col for col in month_cols if col.startswith(year)]

    고용률[year] = df[year_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)


# 성별 데이터 버리고 합계로
고용률 = 고용률.loc[고용률["성별"] == "계", :]

# 합계 데이터만 있으므로, 성별 컬럼도 버림
고용률.drop(columns="성별", inplace=True)

# 시도별이 "계"라고 되어있는걸 "전국"으로 통일
고용률.loc[0, "시도별"] = "전국"

# wide -> long
고용률 = 고용률.melt(
    id_vars="시도별",
    var_name="연도",
    value_name="고용률" 
)

# 정수형으로
고용률["연도"] = pd.to_numeric(고용률["연도"])
고용률["고용률"] = pd.to_numeric(고용률["고용률"])

# 지역명 표준화
고용률.loc[고용률["시도별"] == "제주도", "시도별"] = "제주특별자치도"

고용률

,시도별,연도,고용률
0,전국,1999,58.042857
1,서울특별시,1999,57.400000
2,부산광역시,1999,54.285714
3,대구광역시,1999,55.042857
4,인천광역시,1999,57.085714
...,...,...,...
499,전북특별자치도,2026,62.700000
500,전라남도,2026,64.575000
501,경상북도,2026,62.375000
502,경상남도,2026,63.350000


In [1690]:
df = pd.read_csv("조혼인율_시도_시_군_구__20260524221114.csv")
df.head()

,행정구역별,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,전국,7.0,6.7,6.3,6.3,6.4,6.5,6.8,7.0,6.6,...,5.5,5.2,5.0,4.7,4.2,3.8,3.7,3.8,4.4,4.7
1,서울특별시,7.7,7.5,7.1,7.1,7.0,7.0,7.3,7.5,7.0,...,5.9,5.5,5.4,5.0,4.7,3.9,3.8,3.9,4.6,5.3
2,부산광역시,6.1,5.8,5.6,5.4,5.4,5.2,5.5,6.0,5.6,...,4.9,4.5,4.3,4.1,3.6,3.3,3.2,3.1,3.5,4.0
3,대구광역시,6.4,5.9,5.4,5.5,5.5,5.2,5.5,5.9,5.5,...,5.0,4.6,4.5,4.1,3.5,3.1,3.2,3.4,3.9,4.0
4,인천광역시,7.1,6.7,6.6,6.1,6.2,6.4,6.6,6.9,6.8,...,5.5,5.2,5.1,4.6,4.0,3.7,3.7,3.9,4.4,4.6


In [1691]:
# 조혼인율데이터 전처리

# wide -> long
조혼인율 = df.melt(
    id_vars="행정구역별",
    var_name="연도",
    value_name="조혼인율"
)

# 컬럼이름 : 행정구역별 -> 시도별
조혼인율 = 조혼인율.rename(columns={
    "행정구역별": "시도별"
})

# 숫자형으로
조혼인율["연도"] = pd.to_numeric(조혼인율["연도"])
조혼인율["조혼인율"] = pd.to_numeric(조혼인율["조혼인율"], errors="coerce")

조혼인율

,시도별,연도,조혼인율
0,전국,2000,7.0
1,서울특별시,2000,7.7
2,부산광역시,2000,6.1
3,대구광역시,2000,6.4
4,인천광역시,2000,7.1
...,...,...,...
489,전라남도,2025,4.1
490,경상북도,2025,3.6
491,경상남도,2025,3.7
492,제주특별자치도,2025,4.2


In [1692]:
df = pd.read_csv("시도별_평균초혼연령_20260525003556.csv", header=[0, 1])
df.head()

시도별   1990          1991          1992          1993          1994  ...  \
     시도별     남편     아내     남편     아내     남편     아내     남편     아내     남편  ...   
0     전국  27.79  24.78  27.91  24.84  28.01  24.93  28.09  25.01  28.21  ...   
1  서울특별시  28.26  25.54  28.38  25.58  28.47  25.63  28.52  25.71  28.62  ...   
2  부산광역시  27.96  25.03  28.07  25.08  28.19  25.23  28.25  25.30  28.38  ...   
3  대구광역시  27.66  24.95  27.78  25.03  27.93  25.11  28.01  25.16  28.07  ...   
4  인천광역시  27.80  24.95  27.90  25.03  28.01  24.98  28.17  25.12  28.23  ...   

    2021          2022          2023          2024          2025         
      남편     아내     남편     아내     남편     아내     남편     아내     남편     아내  
0  33.35  31.08  33.72  31.26  33.97  31.45  33.86  31.55  33.85  31.62  
1  33.85  31.91  34.17  32.15  34.38  32.42  34.32  32.44  34.18  32.40  
2  33.40  31.30  33.94  31.67  34.31  31.95  34.16  32.04  34.00  31.99  
3  33.32  31.03  33.63  31.22  33.71  31.43  33.74  31.54  33.77  31.61  
4  33.21  30.95  33.63  31.29  33.79  31.57  33.74  31.67  33.93  31.82  

[5 rows x 73 columns]

In [1693]:
# 컬럼 레벨 이름 지정
df.columns.names = ["연도", "성별"]

# 첫 번째 컬럼은 ('시도별', '시도별') 같은 형태이므로 따로 잡아둠
id_col = df.columns[0]

평균초혼연령 = (
    df
    .set_index(id_col)
    .rename_axis(index="시도별")
    .stack(level="연도", future_stack=True)
    .reset_index()      # stack()을 한 뒤에는 "시도별"과 "연도"가 인덱스에 들어가있음 -> 다시 일반 컬럼으로 꺼냄........ stack() 직후에는 인덱스가 두 개짜리 MultiIndex가 됨 -> reset_index를 하면 시도별과 연도를 둘 다 일반 컬럼으로 꺼냄
    .rename(columns={
        "남편": "평균초혼연령(남편)",
        "아내": "평균초혼연령(아내)"
    })
)

# 컬럼명 위에 남는 name 제거
평균초혼연령.columns.name = None

cols = ["연도", "평균초혼연령(남편)", "평균초혼연령(아내)"]

평균초혼연령[cols] = 평균초혼연령[cols].apply(
    pd.to_numeric,
    errors="coerce"
)

평균초혼연령


,시도별,연도,평균초혼연령(남편),평균초혼연령(아내)
0,전국,1990,27.79,24.78
1,전국,1991,27.91,24.84
2,전국,1992,28.01,24.93
3,전국,1993,28.09,25.01
4,전국,1994,28.21,25.14
...,...,...,...,...
679,국외,2021,32.77,29.63
680,국외,2022,33.03,28.74
681,국외,2023,33.53,28.38
682,국외,2024,33.56,28.57


In [1694]:
df = pd.read_csv("아파트_매매_실거래가격지수_20260525142438.csv")
df.head()

,행정구역별(1),행정구역별(2),2006.01,2006.02,2006.03,2006.04,2006.05,2006.06,2006.07,2006.08,...,2025.06,2025.07,2025.08,2025.09,2025.10,2025.11,2025.12,2026.01,2026.02,2026.03 p)
0,행정구역별(1),행정구역별(2),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),...,지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),지수 (2017.11 = 100.0),잠정 증감률 (%)
1,전국,소계,60.0,60.5,61.2,61.7,62.0,62.5,62.6,63.5,...,125.7,125.9,125.9,126.7,127.5,128.1,128.4,129.4,130.4,-0.50
2,수도권,소계,62.8,63.7,64.8,65.6,66.2,66.5,66.8,67.8,...,150.9,151.6,151.4,153.0,154.5,155.4,156.1,157.8,159.8,-0.64
3,지방,소계,57.7,57.6,57.7,57.7,57.7,58.4,58.3,58.8,...,104.7,104.6,104.7,105.0,105.1,105.4,105.5,105.9,106.0,-0.31
4,서울,서울,58.5,59.4,60.9,61.6,61.9,61.8,62.2,63.0,...,179.8,181.6,181.5,185.3,188.5,191.1,191.9,194.7,198.4,-0.59


In [1695]:
# 아파트 매매 실거래가격지수 데이터 전처리

아파트매매실거래가격지수 = df.drop(index=0).reset_index(drop=True)
아파트매매실거래가격지수 = 아파트매매실거래가격지수.drop(columns=["2026.03 p)"], errors="ignore")

# 월별 지수 컬럼 지정
month_cols = 아파트매매실거래가격지수.columns.drop(["행정구역별(1)", "행정구역별(2)"])

# 3. wide → long
아파트매매실거래가격지수 = 아파트매매실거래가격지수.melt(
    id_vars=["행정구역별(1)", "행정구역별(2)"],
    value_vars=month_cols,
    var_name="년월",
    value_name="아파트매매실거래가격지수"
)

아파트매매실거래가격지수["아파트매매실거래가격지수"] = pd.to_numeric(
    아파트매매실거래가격지수["아파트매매실거래가격지수"],
    errors="coerce"
)

아파트매매실거래가격지수["연도"] = 아파트매매실거래가격지수["년월"].str[:4].astype(int)

region_map = {
    "서울": "서울특별시",
    "부산": "부산광역시",
    "대구": "대구광역시",
    "인천": "인천광역시",
    "광주": "광주광역시",
    "대전": "대전광역시",
    "울산": "울산광역시",
    "세종": "세종특별자치시",
    "경기": "경기도",
    "강원": "강원특별자치도",
    "충북": "충청북도",
    "충남": "충청남도",
    "전북": "전북특별자치도",
    "전남": "전라남도",
    "경북": "경상북도",
    "경남": "경상남도",
    "제주": "제주특별자치도",
}

아파트매매실거래가격지수["시도별"] = 아파트매매실거래가격지수["행정구역별(2)"].map(region_map)

# 전국 행은 행정구역별(2)가 '소계'라서 따로 처리
아파트매매실거래가격지수.loc[아파트매매실거래가격지수["행정구역별(1)"].eq("전국"), "시도별"] = "전국"

# 기존 데이터프레임에 있는 지역만 남김
아파트매매실거래가격지수 = 아파트매매실거래가격지수[아파트매매실거래가격지수["시도별"].isin(아파트매매실거래가격지수["시도별"].unique())]

아파트매매실거래가격지수 = (
    아파트매매실거래가격지수
    .groupby(["시도별", "연도"], as_index=False)["아파트매매실거래가격지수"]
    .mean()
    .rename(columns={
        "아파트매매실거래가격지수": "아파트매매실거래가격지수_연평균"
    })
)

아파트매매실거래가격지수["연도"] = pd.to_numeric(아파트매매실거래가격지수["연도"])
아파트매매실거래가격지수["아파트매매실거래가격지수_연평균"] = pd.to_numeric(아파트매매실거래가격지수["아파트매매실거래가격지수_연평균"])

아파트매매실거래가격지수

,시도별,연도,아파트매매실거래가격지수_연평균
0,강원특별자치도,2006,58.600000
1,강원특별자치도,2007,58.841667
2,강원특별자치도,2008,60.275000
3,강원특별자치도,2009,61.458333
4,강원특별자치도,2010,64.416667
...,...,...,...
352,충청북도,2022,112.608333
353,충청북도,2023,107.825000
354,충청북도,2024,109.041667
355,충청북도,2025,108.158333


In [1696]:
df = pd.read_csv(
    "주택의_종류별_주택__읍면동_연도_끝자리_0__5___시군구_그_외_연도__20260525145619.csv",
    encoding="utf-8-sig",
    header=[0, 1]
)

df.head()

행정구역별(읍면동)      2015                                                        \
  행정구역별(읍면동)        주택   단독주택-계  단독주택-일반 단독주택-다가구 단독주택-영업겸용      아파트    연립주택   
0         전국  16367006  3973961  2714279   865154    394528  9806062  485349   
1         읍부   1614808   544351   467388    38561     38402   864404   66387   
2         면부   1982064  1524970  1448705    32889     43376   334810   31485   
3         동부  12770134  1904640   798186   793704    312750  8606848  387477   
4      서울특별시   2793244   355039    88163   217616     49260  1636896  117235   

                         ...         2023      2024                    \
     다세대주택 비주거용 건물 내 주택  ... 비주거용 건물 내 주택        주택   단독주택-계  단독주택-일반   
0  1898090       203544  ...       211551  19872674  3841487  2621805   
1   112764        26902  ...        31645   2176913   586641   490362   
2    61366        29433  ...        32506   2163044  1549610  1453444   
3  1723960       147209  ...       147400  15532717  1705236   677999   
4   654372        29702  ...        27301   3170332   286542    58667   

                                                              
  단독주택-다가구 단독주택-영업겸용       아파트    연립주택    다세대주택 비주거용 건물 내 주택  
0   771877    447805  12973943  542861  2302915       211468  
1    44896     51383   1327806   83683   147011        31772  
2    38174     57992    467415   38756    74566        32697  
3   688807    338430  11178722  420422  2081338       146999  
4   177740     50135   1906060  110623   839901        27206  

[5 rows x 91 columns]

In [1697]:
# 주택수 데이터 전처리

# ("(행정구역별(읍면동)", "행정구역별(읍면동)")
region_col = df.columns[0]

# wide 형태 → long 형태
주택수 = (
    df
    .set_index(region_col)
    .stack(level=0, future_stack=True)
    .reset_index()
    .rename(columns={
        region_col: "시도별",
        "level_1": "연도"
    })
)

# 컬럼 정리
주택수["시도별"] = 주택수["시도별"].astype(str).str.strip()

# 읍부, 면부, 동부 제거
주택수 = 주택수[
    ~주택수["시도별"].isin(["읍부", "면부", "동부"])
].copy()

# 주택 컬럼만 선택
주택수 = 주택수[["시도별", "연도", "주택"]].copy()

# 숫자형 변환
# 주택수["연도"] = pd.to_numeric(주택수["연도"], errors="coerce")
# 주택수["주택"] = pd.to_numeric(주택수["주택"], errors="coerce")

# 컬럼명 변경
주택수 = 주택수.rename(columns={"주택": "주택수"})

주택수

,시도별,연도,주택수
0,전국,2015,16367006
1,전국,2016,16692230
2,전국,2017,17122573
3,전국,2018,17633327
4,전국,2019,18126954
...,...,...,...
205,제주특별자치도,2020,246451
206,제주특별자치도,2021,249629
207,제주특별자치도,2022,252863
208,제주특별자치도,2023,257878


In [1698]:
df = pd.read_csv("시도별_경력단절여성_20260525151911.csv", header=[0, 1], index_col=0)
df.head()

행정구역별               2014                                        2015  \
행정구역별 15 - 54세 기혼여성 (천명) 미취업여성 (천명) - 경력단절여성 (천명) 15 - 54세 기혼여성 (천명)   
전국                  9733       3957          2164               9561   
서울특별시               1819        732           358               1791   
부산광역시                618        261           117                603   
대구광역시                475        205           113                459   
인천광역시                587        243           122                570   

행정구역별                                        2016                           \
행정구역별 미취업여성 (천명) - 경력단절여성 (천명) 15 - 54세 기혼여성 (천명) 미취업여성 (천명) - 경력단절여성 (천명)   
전국          3863          2073               9376       3727          1924   
서울특별시        745           369               1750        716           348   
부산광역시        255           117                583        242           115   
대구광역시        175           100                445        170            85   
인천광역시        240           121                566        228           114   

행정구역별               2017  ...          2021               2022             \
행정구역별 15 - 54세 기혼여성 (천명)  ... - 경력단절여성 (천명) 15 - 54세 기혼여성 (천명) 미취업여성 (천명)   
전국                  9159  ...          1448               8103       3027   
서울특별시               1684  ...           239               1368        504   
부산광역시                557  ...            78                475        187   
대구광역시                433  ...            77                366        134   
인천광역시                550  ...            75                482        189   

행정구역별                             2023                           \
행정구역별 - 경력단절여성 (천명) 15 - 54세 기혼여성 (천명) 미취업여성 (천명) - 경력단절여성 (천명)   
전국             1397               7943       2837          1349   
서울특별시           220               1323        467           180   
부산광역시            80                458        171            84   
대구광역시            69                366        130            69   
인천광역시            81                468        172            78   

행정구역별               2024                           
행정구역별 15 - 54세 기혼여성 (천명) 미취업여성 (천명) - 경력단절여성 (천명)  
전국                  7654       2601          1215  
서울특별시               1267        397           171  
부산광역시                440        154            71  
대구광역시                349        128            66  
인천광역시                456        164            81  

[5 rows x 33 columns]

In [1699]:
# 컬럼 레벨 이름 지정
df.columns.names = ["연도", "구분"]

# "15 - 54세 기혼여성 (천명)" 컬럼 제거
df = df.drop(
    columns="15 - 54세 기혼여성 (천명)",
    level="구분"
)

# 컬럼 정리: 연도는 int로, "- 경력단절여성"의 앞 '- ' 제거
df.columns = pd.MultiIndex.from_tuples(
    [
        (int(year), str(col).replace("- ", "").strip())
        for year, col in df.columns
    ],
    names=["연도", "구분"]
)

# wide → long 형태로 변환
df = (
    df
    .stack(level="연도", future_stack=True)
    .reset_index()
    .rename(columns={"level_0": "시도별"})
)

# 숫자형 변환
for col in df.columns.difference(["시도별", "연도"]):
    df[col] = pd.to_numeric(df[col], errors="coerce")


# 지역명 맞추기
df["시도별"] = df["시도별"].replace({
    "전라북도": "전북특별자치도",
    "강원도": "강원특별자치도"
})

merged = merged.merge(
    df,
    on=["시도별", "연도"],
    how="left"
)

merged

,시도별,연도,합계출산율,1인당 지역총소득,1인당 개인소득,고용률,미취업여성 (천명),경력단절여성 (천명)
0,전국,2000,1.480,13860,8694,58.508333,NaN,NaN
1,서울특별시,2000,1.275,17194,9978,58.250000,NaN,NaN
2,부산광역시,2000,1.235,10933,8296,55.366667,NaN,NaN
3,대구광역시,2000,1.378,10684,8274,55.991667,NaN,NaN
4,인천광역시,2000,1.473,10633,7651,58.175000,NaN,NaN
...,...,...,...,...,...,...,...,...
369,전북특별자치도,2021,0.850,31803,20745,61.241667,91.0,34.0
370,전라남도,2021,1.017,37741,20779,64.616667,84.0,38.0
371,경상북도,2021,0.966,38071,20522,61.008333,167.0,70.0
372,경상남도,2021,0.903,33363,20535,60.666667,209.0,85.0


In [1700]:
df = pd.read_csv("학교급_및_시도별__학생_1인당_월평균_사교육비_20260525155257.csv", header=[0, 1])

df.head()

시도별      2009                                            2010            \
     시도별 평  균 (만원) 초등학교 (만원) 중학교 (만원) 고등학교 (만원) 일반고 (만원) 평  균 (만원) 초등학교 (만원)   
0     전국      24.2      24.5     26.0      21.7     26.9      24.0      24.5   
1  서울특별시      33.1      30.5     32.6      37.7     43.3      32.1      29.6   
2  부산광역시      20.3      19.7     21.8      19.8     25.1      20.8      19.5   
3  대구광역시      25.1      25.3     29.1      20.6     23.5      25.0      23.2   
4  인천광역시      22.1      23.1     21.3      21.1     25.3      22.0      23.3   

                      ...      2024                                        \
  중학교 (만원) 고등학교 (만원)  ... 평  균 (만원) 초등학교 (만원) 중학교 (만원) 고등학교 (만원) 일반고 (만원)   
0     25.5      21.8  ...      47.4      44.2     49.0      52.0     58.7   
1     30.9      37.2  ...      67.3      60.9     69.1      76.9     85.2   
2     22.8      20.7  ...      48.3      46.2     51.8      48.7     57.5   
3     31.2      21.8  ...      47.8      45.8     48.6      50.9     60.1   
4     22.0      19.8  ...      45.9      42.4     47.7      50.8     58.2   

       2025                                        
  평  균 (만원) 초등학교 (만원) 중학교 (만원) 고등학교 (만원) 일반고 (만원)  
0      45.8      43.3     46.1      49.9     56.0  
1      66.3      60.1     66.4      76.7     83.9  
2      45.6      45.3     47.4      44.0     52.3  
3      44.7      40.8     47.4      49.0     57.7  
4      43.8      39.8     44.6      50.5     58.2  

[5 rows x 86 columns]

In [1701]:
# 시도별을 index로 설정
df = df.set_index(("시도별", "시도별"))
df.index.name = "시도별"

# 컬럼 레벨 이름 지정
df.columns.names = ["연도", "구분"]

# "평균(만원)" 컬럼만 선택
df = df.loc[
    :,
    df.columns.get_level_values("구분").str.replace(" ", "", regex=False) == "평균(만원)"
]

# wide → long 변환
df = (
    df
    .stack(level="연도", future_stack=True)
    .reset_index()
)

df.columns.name = None

# 컬럼명 정리
df = df.rename(columns={
    "평  균 (만원)": "사교육비_평균(만원)",
    "평균(만원)": "사교육비_평균(만원)"
})

# 혹시 공백 때문에 컬럼명이 다르게 남았을 경우 처리
df.columns = [
    "사교육비_평균(만원)" if str(col).replace(" ", "") == "평균(만원)" else col
    for col in df.columns
]

# 연도 숫자형 변환
df["연도"] = df["연도"].astype(int)

# 값 숫자형 변환
df["사교육비_평균(만원)"] = pd.to_numeric(
    df["사교육비_평균(만원)"],
    errors="coerce"
)

# 지역명 맞추기
df["시도별"] = df["시도별"].replace({
    "전라북도": "전북특별자치도"
})

# 기존 데이터와 병합
merged["연도"] = merged["연도"].astype(int)

merged = merged.merge(
    df,
    on=["시도별", "연도"],
    how="left"
)

merged

,시도별,연도,합계출산율,1인당 지역총소득,1인당 개인소득,고용률,미취업여성 (천명),경력단절여성 (천명),사교육비_평균(만원)
0,전국,2000,1.480,13860,8694,58.508333,NaN,NaN,NaN
1,서울특별시,2000,1.275,17194,9978,58.250000,NaN,NaN,NaN
2,부산광역시,2000,1.235,10933,8296,55.366667,NaN,NaN,NaN
3,대구광역시,2000,1.378,10684,8274,55.991667,NaN,NaN,NaN
4,인천광역시,2000,1.473,10633,7651,58.175000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
369,전북특별자치도,2021,0.850,31803,20745,61.241667,91.0,34.0,27.4
370,전라남도,2021,1.017,37741,20779,64.616667,84.0,38.0,23.3
371,경상북도,2021,0.966,38071,20522,61.008333,167.0,70.0,27.9
372,경상남도,2021,0.903,33363,20535,60.666667,209.0,85.0,27.6


In [1702]:
df = pd.read_csv("소비자물가지수_시도__20260525160029.csv")

# 불필요한 시도별(1) 제거, 시도별(2)를 시도별로 사용
df = df.drop(columns=["시도별(1)"])
df = df.rename(columns={"시도별(2)": "시도별"})

# 소계 -> 전국 이름 맞추기
df["시도별"] = df["시도별"].replace({
    "소계": "전국"
})

# wide → long 변환
df = df.melt(
    id_vars="시도별",
    var_name="연월",
    value_name="소비자물가지수"
)

# "-"를 결측치로 처리하고 숫자형 변환
df["소비자물가지수"] = pd.to_numeric(
    df["소비자물가지수"].replace("-", pd.NA),
    errors="coerce"
)

# 연도 추출
df["연도"] = df["연월"].str[:4].astype(int)

# 연도별 평균 소비자물가지수 계산
df = (
    df
    .groupby(["시도별", "연도"], as_index=False)["소비자물가지수"]
    .mean()
)

# 지역명 통일
region_map = {
    "강원도": "강원특별자치도",
    "전라북도": "전북특별자치도"
}

df["시도별"] = df["시도별"].replace(region_map)
merged["시도별"] = merged["시도별"].replace(region_map)

# 연도 타입 맞추기
merged["연도"] = merged["연도"].astype(int)

# 병합
merged = merged.merge(
    df,
    on=["시도별", "연도"],
    how="left"
)


# merged = merged[merged["시도별"] != "세종특별자치시"]
# merged = merged.drop(columns=["소비자물가지수", "아파트매매실거래가격지수_연평균"])
merged[merged.isna().any(axis=1)]

merged.to_csv("merged2.csv", encoding="utf-8-sig")

merged

,시도별,연도,합계출산율,1인당 지역총소득,1인당 개인소득,고용률,미취업여성 (천명),경력단절여성 (천명),사교육비_평균(만원),소비자물가지수
0,전국,2000,1.480,13860,8694,58.508333,NaN,NaN,NaN,63.151000
1,서울특별시,2000,1.275,17194,9978,58.250000,NaN,NaN,NaN,61.828333
2,부산광역시,2000,1.235,10933,8296,55.366667,NaN,NaN,NaN,62.709500
3,대구광역시,2000,1.378,10684,8274,55.991667,NaN,NaN,NaN,63.668333
4,인천광역시,2000,1.473,10633,7651,58.175000,NaN,NaN,NaN,63.584917
...,...,...,...,...,...,...,...,...,...,...
369,전북특별자치도,2021,0.850,31803,20745,61.241667,91.0,34.0,27.4,102.590000
370,전라남도,2021,1.017,37741,20779,64.616667,84.0,38.0,23.3,102.617500
371,경상북도,2021,0.966,38071,20522,61.008333,167.0,70.0,27.9,102.738333
372,경상남도,2021,0.903,33363,20535,60.666667,209.0,85.0,27.6,102.534167


In [1703]:
# 데이터프레임 병합

step1 = pd.merge(
    합계출산율, 
    일인당지역총소득및개인소득,
    on=["시도별", "연도"], how="inner"
)

step3 = pd.merge(step2, 고용률,
                 on=["시도별", "연도"], how="inner")

merged = step3

merged


,시도별,연도,합계출산율,1인당 지역총소득,1인당 개인소득,고용률
0,전국,2000,1.480,13860,8694,58.508333
1,서울특별시,2000,1.275,17194,9978,58.250000
2,부산광역시,2000,1.235,10933,8296,55.366667
3,대구광역시,2000,1.378,10684,8274,55.991667
4,인천광역시,2000,1.473,10633,7651,58.175000
...,...,...,...,...,...,...
369,전북특별자치도,2021,0.850,31803,20745,61.241667
370,전라남도,2021,1.017,37741,20779,64.616667
371,경상북도,2021,0.966,38071,20522,61.008333
372,경상남도,2021,0.903,33363,20535,60.666667
